# 03 · Modelos y líneas base

**Objetivo:** revisar el último modelo entrenado: qué candidato se eligió, si supera a la línea
base y en qué segmentos acierta o falla.

**Requiere:** `contrataciones-ml train` (guarda el modelo en `models/trained`, la bitácora en
`models/logs` y el dataset en `data/features/dataset_modelado.parquet`).

| Modelo | Qué es |
| --- | --- |
| Adjudicado = referencial | Supone que se adjudica exactamente el valor referencial |
| Mediana global | La misma razón para todos los procesos |
| Línea base B1 | Mediana de la razón por método y categoría (criterio de aceptación) |
| Ridge | Lineal con variables codificadas: interpretable |
| HistGradientBoosting | Árboles con interacciones; pérdida absoluta (mediana condicional) |

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"          # capa gold
FEATURES = ROOT / "data" / "features"        # dataset de modelado
PREDICTIONS = ROOT / "data" / "predictions"  # monto esperado y atípicos
MODELS = ROOT / "models"                     # modelos entrenados
RAW_API = ROOT / "data" / "raw" / "api"      # páginas crudas de la API del OECE
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
COLORES_DESVIACION = {
    "Normal": "#C8CFD8",
    "Sobrecosto inusual": "#B5533C",
    "Subcosto inusual": "#3E6A8A",
    "Revisar datos": "#C9A227",
}
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
porcentaje = mticker.PercentFormatter(1.0)
print("Proyecto:", ROOT)

In [ ]:
import json

from contrataciones.models.registry import load_model

modelo = load_model(MODELS)
bitacora = json.loads((MODELS / "logs" / f"experimento_{modelo['version']}.json").read_text(encoding="utf-8"))
datos = modelo["data"]
print(f"Modelo {modelo['version']} | seleccionado: {modelo['selected']} | aceptado: {modelo['accepted']}")
print(f"Entrenado con la corrida {datos.get('run_id')} (fuente {datos.get('fuente')})")
if datos.get("sinteticos"):
    print("Datos SINTÉTICOS: las métricas no dicen nada de la calidad del modelo.")
pd.Series(modelo["split_dates"])

## 1. Validación: elección del candidato

Se elige el candidato con menor MAE en escala log en la partición de validación.

In [ ]:
validacion = pd.DataFrame(modelo["metrics"]["validacion"]).T.sort_values("mae_log")
validacion

## 2. Prueba: ¿supera a la línea base?

La partición de prueba se usa una sola vez. El modelo se acepta si reduce el MAE (log) de la
línea base en al menos el criterio configurado.

* `exp(mae_log) - 1` es el error relativo típico.
* `r2_log_ratio` es el R² honesto. `r2_log_monto` sale cercano a 1 **solo** porque el valor
  referencial explica la escala del monto: no lo uses para juzgar el modelo.

In [ ]:
prueba = pd.DataFrame(modelo["metrics"]["prueba"]).T.sort_values("mae_log")
ax = prueba["mae_log"].sort_values(ascending=False).plot.barh(title="MAE (log) en prueba", color=PALETA[1])
ax.axvline(prueba.loc["linea_base", "mae_log"], color=PALETA[4], linestyle="--", label="línea base B1")
ax.legend(); ax.set_ylabel(""); plt.show()
mejora = modelo["metrics"]["mejora_vs_linea_base"]
criterio = modelo["metrics"]["criterio_aceptacion"]
print(f"Mejora frente a la línea base: {mejora:.1%} (criterio >= {criterio:.0%})")
prueba

## 3. Métricas por segmento

Un buen promedio puede esconder segmentos donde el modelo es peor que la línea base.

In [ ]:
segmentos = pd.DataFrame(bitacora["metricas_por_segmento"])
for dimension, grupo in segmentos.groupby("dimension", sort=False):
    print(f"\n{dimension}")
    display(grupo.drop(columns="dimension").set_index("segmento").style.format(
        {"mae_log_modelo": "{:.4f}", "mae_log_linea_base": "{:.4f}", "pct_mejora": "{:.1%}"}))

## 4. Importancia de las variables

Cuánto empeora el MAE (log) en prueba al desordenar cada variable. Valores cercanos a 0 (o
negativos) indican que el modelo no depende de esa variable.

In [ ]:
importancia = pd.DataFrame(modelo["importance"]).set_index("variable")
ax = importancia["aumento_mae_log"].sort_values().plot.barh(
    xerr=importancia["desviacion"], title="Importancia por permutación", color=PALETA[0])
ax.set_ylabel(""); plt.show()

## 5. Residuos fuera de muestra

Cada proceso del histórico se predijo con un modelo que no lo vio (por trimestres). Con estos
residuos se calibra la escala robusta de cada segmento que usan los montos atípicos.

In [ ]:
dataset = pd.read_parquet(FEATURES / "dataset_modelado.parquet")
entrenables = dataset[dataset["es_entrenable"]]
residuo = entrenables["log_ratio_adjudicado"] - entrenables["prediccion_fuera_de_muestra"]
for categoria, valores in residuo.groupby(entrenables["categoria"]):
    plt.hist(valores, bins=50, alpha=0.5, label=categoria)
plt.title("Residuos fuera de muestra por categoría"); plt.xlabel("log(real) - log(esperado)")
plt.legend(); plt.show()
pd.DataFrame(modelo["scales"]).T

## Conclusiones

* Si el modelo **no** supera a la línea base, el monto esperado se apoya sobre todo en la
  mediana de cada segmento: los atípicos siguen siendo útiles, pero con menos matices.
* Con datos sintéticos el resultado esperado es *no aceptado*: la razón es ruido.
* Revisa los segmentos con mejora negativa antes de usar el modelo con datos reales.
* Siguiente paso: `contrataciones-ml predict` y `04_evaluacion_y_atipicos.ipynb`.